# CN Lab - Python Experiments
Every program is plain Python on `127.0.0.1` (same PC). Use **two terminals**: server first, then client. In the lab, type each block into the file named above it.

| # | Experiment | Appears in | Files |
|---|---|---|---|
| 1 | TCP quiz: 5 MCQs, server scores | Q2a | `server.py`, `client.py` |
| 2 | TCP multi-client exam | older lab sheet (Exp 4) | `server.py`, `client.py` |
| 3 | UDP sentiment | Q4b, Q10a | `server.py`, `client.py` |
| 4 | Mininet, no controller | Q5a, Q8b | `topology.py` |
| 5 | Mininet, with controller | older lab sheet (Exp 9) | `topology.py` |

**Before each run:** `cd` into a folder for that experiment so the files don't mix. **Stop a server:** Ctrl+C.

**Common errors**
| Error | Cause / fix |
|---|---|
| `ConnectionRefusedError` | Server not running. Start `server.py` first |
| `Address already in use` | Old server still running or port in TIME_WAIT. Close it, wait a few seconds, or change the port |
| UDP client hangs | Server not running (UDP gives no error). Start the server |
| `python3: can't open file` | Wrong folder. Run `ls` and `cd` to it |
| Mininet `File exists` / stale errors | `sudo mn -c`, then run again |
| Mininet `Permission denied` | Use `sudo` |

## 1. TCP quiz (Q2a)
Server sends 5 MCQs one at a time, client answers each, server checks them and prints the final score.

**server.py**
```python
import socket

QUESTIONS = [
    ("Q1: Which layer does routing? A) Physical B) Network C) Transport D) Session", "B"),
    ("Q2: TCP is? A) Connectionless B) Connection-oriented C) Unreliable D) None", "B"),
    ("Q3: Default HTTP port? A) 21 B) 25 C) 80 D) 443", "C"),
    ("Q4: IPv4 address size? A) 16 B) 32 C) 64 D) 128 bits", "B"),
    ("Q5: Which is a Layer 2 device? A) Hub B) Switch C) Repeater D) Router", "B"),
]

s = socket.create_server(("127.0.0.1", 5000))
c, addr = s.accept()

score = 0
for q, ans in QUESTIONS:
    c.send(q.encode())
    if c.recv(1024).decode().upper() == ans:
        score += 1

c.send(f"Your score: {score}/{len(QUESTIONS)}".encode())
print("Final score:", score, "/", len(QUESTIONS))
c.close()
s.close()
```

**client.py**
```python
import socket

c = socket.create_connection(("127.0.0.1", 5000))

for _ in range(5):
    print(c.recv(1024).decode())
    c.send(input("Answer: ").encode())

print(c.recv(1024).decode())
c.close()
```

**Run:** terminal 1 `python3 server.py`, terminal 2 `python3 client.py`, type A/B/C/D.

**Change the questions:** edit the `QUESTIONS` list. Each item is `("question text", "correct letter")`. If you change how many there are, change `range(5)` in the client.

**Viva**
- TCP vs UDP? TCP is connection-oriented, reliable and ordered. UDP is connectionless and unreliable.
- Server call order? `socket, bind, listen, accept, recv/send, close`. `create_server` does the first three.
- Client call order? `socket, connect, send/recv, close`. `create_connection` does the first two.
- Three-way handshake? SYN, SYN-ACK, ACK.
- Why `encode()` / `decode()`? Sockets send bytes, not strings.
- What is 127.0.0.1? Loopback (localhost), the same PC.
- `SOCK_STREAM`, `AF_INET`? TCP stream socket, IPv4 family.

## 2. TCP multi-client exam
Many students connect at once. The server starts one thread per student, asks 3 questions and prints each student's answers.

**server.py**
```python
import socket, threading

QUESTIONS = [
    "Q1: Which layer handles routing? (A: Transport, B: Network, C: Data Link)",
    "Q2: TCP is connection-oriented? (A: True, B: False)",
    "Q3: Default HTTP port? (A: 21, B: 80, C: 443)",
]

def handle(conn):
    name = conn.recv(1024).decode()
    answers = []
    for q in QUESTIONS:
        conn.send(q.encode())
        answers.append(conn.recv(1024).decode())
    conn.send("DONE".encode())
    conn.close()
    print(name, "answered:", answers)

s = socket.create_server(("127.0.0.1", 5000))
print("Waiting for students...")
while True:
    conn, addr = s.accept()
    threading.Thread(target=handle, args=(conn,)).start()
```

**client.py**
```python
import socket

c = socket.create_connection(("127.0.0.1", 5000))
c.send(input("Enter your name: ").encode())

while True:
    data = c.recv(1024).decode()
    if data == "DONE":
        print("Exam submitted")
        break
    print(data)
    c.send(input("Your answer: ").encode())

c.close()
```

**Run:** terminal 1 `python3 server.py`, then open terminals 2 and 3 and run `python3 client.py` in each.

**Viva**
- Why threads? `accept()` and `recv()` block, so one thread per client lets all students be served together.
- What does `accept()` return? A new socket for that client and the client's address.
- Why does the server wait for the answer before sending the next question? TCP is a byte stream with no message boundaries, so strict send/receive alternation keeps messages separate.

## 3. UDP sentiment (Q4b, Q10a)
Client sends text, server counts positive and negative words and replies Positive, Negative or Neutral.

**server.py**
```python
import socket

POS = ["good", "great", "happy", "excellent", "love", "amazing", "nice", "wonderful", "best", "fantastic"]
NEG = ["bad", "terrible", "sad", "hate", "worst", "awful", "poor", "horrible", "angry", "disappointing"]

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.bind(("127.0.0.1", 6000))
print("UDP server waiting...")

while True:
    data, addr = s.recvfrom(1024)
    text = data.decode().lower()
    p = sum(w in text for w in POS)
    n = sum(w in text for w in NEG)
    result = "Positive" if p > n else "Negative" if n > p else "Neutral"
    print(text, "->", result)
    s.sendto(result.encode(), addr)
```

**client.py**
```python
import socket

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

while True:
    text = input("Enter text (exit to quit): ")
    if text == "exit":
        break
    s.sendto(text.encode(), ("127.0.0.1", 6000))
    print("Sentiment:", s.recvfrom(1024)[0].decode())

s.close()
```

**Run:** terminal 1 `python3 server.py`, terminal 2 `python3 client.py`. Type `exit` to quit the client.

**Change the word lists:** edit `POS` and `NEG`.

**Viva**
- Why no `listen`, `accept` or `connect`? UDP is connectionless.
- What do `sendto` and `recvfrom` do? Send or receive one datagram together with the other side's address.
- How does the server know where to reply? `recvfrom` returns the sender's address.
- `SOCK_DGRAM`? UDP socket type.
- Is delivery guaranteed? No: no ordering and no retransmission.
- Where is UDP used? DNS, DHCP, VoIP, streaming.
- How does the classifier work? It counts keywords (rule-based), not real NLP. Substring matching means "badminton" would count as negative.

## 4. Mininet, no controller (Q5a, Q8b)
Topology: h1, h2 connect to s1; s1 connects to s2; h3, h4 connect to s2. Hosts are 10.0.0.1 to 10.0.0.4. `failMode='standalone'` makes the switches behave as normal learning switches, so no controller is needed.

**topology.py**
```python
from mininet.net import Mininet
from mininet.cli import CLI

net = Mininet(controller=None)

s1 = net.addSwitch('s1', failMode='standalone')
s2 = net.addSwitch('s2', failMode='standalone')

h1 = net.addHost('h1', ip='10.0.0.1/24')
h2 = net.addHost('h2', ip='10.0.0.2/24')
h3 = net.addHost('h3', ip='10.0.0.3/24')
h4 = net.addHost('h4', ip='10.0.0.4/24')

net.addLink(h1, s1)
net.addLink(h2, s1)
net.addLink(s1, s2)
net.addLink(h3, s2)
net.addLink(h4, s2)

net.start()
net.pingAll()
CLI(net)
net.stop()
```

**Run (Ubuntu lab PC):**
```
sudo python3 topology.py
mininet> nodes
mininet> net
mininet> pingall
mininet> h1 ping -c 3 h4
mininet> exit
sudo mn -c
```
`nodes` and `net` display the topology. `pingall` should show 0% dropped. `net.stop()` and `sudo mn -c` are the clean-up.

**Viva**
- What is Mininet? An emulator that creates virtual hosts, switches and links on one machine.
- Why `controller=None`? There is no OpenFlow controller; the switches run standalone.
- What does `pingall` do? Pings between every pair of hosts and reports the percentage lost.
- What happens with normal OVS and no controller? No flows are installed, so pings fail.
- What is SDN or OpenFlow? Control plane separated from the data plane.
- Why `sudo`? Creating network namespaces needs root.
- Default Mininet subnet? 10.0.0.0/8.

## 5. Mininet, with a controller
Same topology, but the switches are managed by controller `c0`.

**topology.py**
```python
from mininet.net import Mininet
from mininet.node import Controller
from mininet.cli import CLI

net = Mininet(controller=Controller)
net.addController('c0')

s1 = net.addSwitch('s1')
s2 = net.addSwitch('s2')

h1 = net.addHost('h1', ip='10.0.0.1/24')
h2 = net.addHost('h2', ip='10.0.0.2/24')
h3 = net.addHost('h3', ip='10.0.0.3/24')
h4 = net.addHost('h4', ip='10.0.0.4/24')

net.addLink(h1, s1)
net.addLink(h2, s1)
net.addLink(s1, s2)
net.addLink(h3, s2)
net.addLink(h4, s2)

net.start()
net.pingAll()
CLI(net)
net.stop()
```

Run it the same way as experiment 4.

**Difference from experiment 4:** this one creates a controller and the switches use it. Experiment 4 has none.

## Quick revision
| | TCP | UDP |
|---|---|---|
| Server | `create_server`, `accept`, `recv/send` | `socket`, `bind`, `recvfrom/sendto` |
| Client | `create_connection`, `send/recv` | `sendto/recvfrom` |
| Many clients | one `Thread` per client | one loop serves everyone |

**Mininet in 6 steps:** `Mininet()`, add switches and hosts, `addLink`, `start()`, `pingAll()` / `CLI(net)`, `stop()`